[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gregwchase/dynatrace-pubsec-summit-2026/blob/main/notebooks/04_signal_and_dashboards.ipynb)

# Lab 4: Surface Insights with Signal & Track Them with Dashboards
**Traces to Insights: A Hands-On Workshop with Arize AX** · Dynatrace Public Sector Summit 2026 · ⏱️ ~20 min

**In this lab you will:**
- Ship the v2 prompt to "production" and send a second batch of traffic
- Use **Signal** to get a ranked list of issues from production traces, with links to the traces behind each one
- Build a **dashboard** for eval pass rate, cost per trace, latency, and guardrail triggers
- Share it with your team

In [ ]:
# --- Setup: works in Google Colab or a local Jupyter environment ---
import os, sys, subprocess

if "google.colab" in sys.modules:
    REPO_DIR = "/content/dynatrace-pubsec-summit-2026"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-q", "https://github.com/gregwchase/dynatrace-pubsec-summit-2026.git", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif not os.path.exists("civic_agent") and os.path.exists("../civic_agent"):
    os.chdir("..")  # running locally from notebooks/

sys.path.insert(0, os.getcwd())
from workshop_utils import load_credentials

PROJECT = load_credentials()  # prompts for keys if they aren't in .env / Colab Secrets

## 1. "Deploy" v2 and send more traffic

Your project already has v1 traffic from Lab 1. Now send the same mix through the v2 prompt so your dashboard shows a before/after. Each trace carries `metadata.prompt_version`, so you can split by it.

In [ ]:
!{sys.executable} scripts/generate_traffic.py --n 45 --prompt-version v2 --project {PROJECT} --seed 11

## 2. Signal: let Arize find the issues 🔦

Reading traces one at a time doesn't scale. **Signal** analyzes your production traces and returns a **ranked list of issues**. Each issue comes with an explanation, investigation details, and links to the specific traces behind it.

1. Open your project in Arize AX and go to **Signal**.
2. Run an analysis over the last 24 hours of traces.
3. Review the ranked issues. Pay attention to:
   - What patterns did it find that our evals **didn't** look for?
   - Does it separate v1 issues from v2 issues?
   - Click into an issue → open the linked traces → confirm it's real.
4. For an issue worth fixing: add its traces to a dataset (→ Lab 3 loop) or turn the pattern into a new evaluator (→ Lab 2 loop).

> 💡 Signal finds the **unknown unknowns**. Evals measure what you already know to look for. You need both: Signal tells you what to measure next, and evals tell you whether the fix worked.

🤔 **Discuss:** which issue would you fix first, and why? Think about **severity × frequency**. One hallucinated eligibility rule can matter more than 20 slightly long answers.

## 3. Score the new traffic

Run the code evals on every trace (they're instant and free). Run the hallucination judge on any trace that doesn't have a score yet (the v2 traffic). Then the dashboard has comparable pass rates for both prompt versions. In production, an **online eval task** does this automatically.

In [ ]:
import re
import pandas as pd
from phoenix.evals import create_evaluator, evaluate_dataframe
from workshop_utils import export_spans, root_spans, cost_per_trace, to_arize_eval_columns, log_evals

spans = export_spans(hours=24)
traces = root_spans(spans)
traces["tools_called_str"] = traces["tool_calls"].apply(lambda t: ", ".join(t) or "(none)")
traces["output"] = traces["output"].fillna("")

@create_evaluator(name="cites_source", kind="code")
def cites_source(output: str, tool_context: str) -> float:
    return 1.0 if "source_url" not in tool_context or "dcs.columbia.example.gov" in output else 0.0

CLAIM = re.compile(r"\b(you|you're|you are|your household|you'd|we|they)\b.{0,40}\b(qualif|eligib)", re.I)

@create_evaluator(name="eligibility_tool_used", kind="code")
def eligibility_tool_used(output: str, tools_called_str: str) -> float:
    return 1.0 if not CLAIM.search(output) or "check_eligibility" in tools_called_str else 0.0

scored = evaluate_dataframe(dataframe=traces, evaluators=[cites_source, eligibility_tool_used], hide_tqdm_bar=True)
log_evals(to_arize_eval_columns(scored, ["cites_source", "eligibility_tool_used"]))

Now the hallucination judge (same rubric as Lab 2) on traces that haven't been judged yet:

In [ ]:
from phoenix.evals import LLM, create_classifier

hallucination = create_classifier(
    name="hallucination",
    llm=LLM(provider="openai", model="gpt-4.1"),
    choices={"factual": 1.0, "hallucinated": 0.0},
    prompt_template="""You are auditing a government benefits assistant. Its answers must be
based ONLY on information returned by its official tools, never on outside knowledge.

[Resident question]: {{input}}
[Tool results the assistant received]: {{tool_context}}
[Assistant answer]: {{output}}

Does the answer state any program, rule, amount, date, process, or recommendation that is
NOT supported by the tool results? General courtesy, a referral to the DCS hotline, or a
request for more information does not count as hallucination.

Respond "hallucinated" if the answer contains unsupported claims, otherwise "factual".""",
)

already = traces["eval.hallucination.label"].notna() if "eval.hallucination.label" in traces else False
unblocked = traces["guardrail_triggered"].fillna("none") == "none"
todo = traces[unblocked & ~already]
print(f"Judging {len(todo)} traces...")
if len(todo):
    judged = evaluate_dataframe(dataframe=todo, evaluators=[hallucination])
    log_evals(to_arize_eval_columns(judged, ["hallucination"]))
    # keep a local copy of the scores for the next step
    traces.loc[todo.index, "eval.hallucination.score"] = judged["hallucination_score"].apply(lambda s: s.get("score") if isinstance(s, dict) else None).values

## 4. Preview the dashboard metrics in pandas

These are the numbers you'll chart in Arize. Computing them here first is a good habit: you'll know what "right" looks like.

In [ ]:
costs = cost_per_trace(spans)
df = traces.merge(costs, on="context.trace_id", how="left")
df["latency_s"] = (pd.to_datetime(df["end_time"]) - pd.to_datetime(df["start_time"])).dt.total_seconds()
for name in ["cites_source", "eligibility_tool_used"]:
    df[name] = scored[f"{name}_score"].apply(lambda s: s.get("score") if isinstance(s, dict) else None).values
df["hallucination_pass"] = pd.to_numeric(df.get("eval.hallucination.score"), errors="coerce")
df["guardrail"] = df["guardrail_triggered"].fillna("none") != "none"

kpis = df.groupby("prompt_version").agg(
    traces=("context.trace_id", "count"),
    hallucination_pass=("hallucination_pass", "mean"),
    cites_source_pass=("cites_source", "mean"),
    eligibility_tool_pass=("eligibility_tool_used", "mean"),
    avg_cost_usd=("cost_usd", "mean"),
    p50_latency_s=("latency_s", "median"),
    p95_latency_s=("latency_s", lambda s: s.quantile(0.95)),
    avg_llm_calls=("llm_calls", "mean"),
    guardrail_rate=("guardrail", "mean"),
)
kpis.style.format({"hallucination_pass": "{:.0%}", "cites_source_pass": "{:.0%}", "eligibility_tool_pass": "{:.0%}", "guardrail_rate": "{:.0%}",
                   "avg_cost_usd": "${:.5f}", "p50_latency_s": "{:.1f}s", "p95_latency_s": "{:.1f}s", "avg_llm_calls": "{:.1f}"})

## 5. Build the dashboard in Arize 📊

**Dashboards → New Dashboard →** name it `CivicAssist – <initials>`. Add these widgets, each scoped to your project:

| Widget | Metric | Group by / filter | Why your team cares |
|---|---|---|---|
| **Eval pass rate** (line) | `eval.hallucination` / `eval.cites_source` pass % | `metadata.prompt_version` | Quality trend; catches regressions after a deploy |
| **Cost per trace** (line) | Token count (prompt + completion) or cost, averaged per trace | `metadata.model` | Budget; ties directly to ROI |
| **Latency** (line) | p50 and p95 latency of root spans | — | Resident experience, SLA tracking |
| **Guardrail triggers** (bar) | Count where `metadata.guardrail_triggered != "none"` | by guardrail | Security/PII posture for compliance reporting |
| **Traffic volume** (bar) | Trace count | by `metadata.tools_called` | Which services residents actually use |
| **Errors** (stat) | Count of spans with `status_code = ERROR` | — | Reliability |

Then:
1. Set the time range to the last few hours and confirm that the v1 → v2 change is visible.
2. **Share** the dashboard (Share button) with a teammate or your neighbor.
3. Optional: create a **monitor** on the eval pass rate (e.g., alert if hallucination pass rate < 90%) so the team hears about it before residents do.

🤔 **Discuss:** if you were briefing a program director, which **two** widgets would you show? What about a security officer?

**Next:** [Lab 5: ROI →](05_roi.ipynb)